In [1]:
import numpy as np
from scipy.io import loadmat

from lib.read_data import u_ref, y_ref

MPCs = ["incremental", "IHMPC"]

mat = [loadmat(f"./MATLAB/{m}/output.mat") for m in MPCs]

u = np.array([mat[i]["uk"].T + u_ref for i in range(len(MPCs))])
y = np.array([mat[i]["yk"].T + y_ref for i in range(len(MPCs))])
y_sp = np.array([mat[i]["ysp"].T + y_ref for i in range(len(MPCs))])
Jk = np.array([mat[i]["Jk"].T for i in range(len(MPCs))])

cycles = np.linspace(0, u[0].shape[0] - 1, u[0].shape[0])

print(u.shape)


(2, 200, 4)


In [2]:
from lib.plot import plt
from lib.read_data import u_labels, y_labels

setpoint_args = {
    "label": "Setpoint",
    "linestyle": "--",
    "color": "black",
    "linewidth": 2,
}

fig, axes = plt.subplots(3, 2, figsize=(16, 12), dpi=300, sharex=True)

# ---- SP ----
axes[1, 1].plot(cycles, y_sp[0][:, 0], **setpoint_args)
axes[2, 0].plot(cycles, y_sp[0][:, 1], **setpoint_args)
axes[2, 1].plot(cycles, y_sp[0][:, 2], **setpoint_args)

for i in range(len(MPCs)):
    # t_feed
    axes[0, 0].plot(cycles, u[i][:, 0], label=f"{u_labels[0]} - {MPCs[i]}")

    # t_rinse
    axes[0, 1].plot(cycles, u[i][:, 1], label=f"{u_labels[1]} - {MPCs[i]}")

    # t_blow + t_purge
    axes[1, 0].plot(cycles, u[i][:, 2], label=f"{u_labels[2]} - {MPCs[i]}")
    axes[1, 0].plot(cycles, u[i][:, 3], label=f"{u_labels[3]} - {MPCs[i]}")

    # Pureza H2
    axes[1, 1].plot(cycles, y[i][:, 0], label=f"{y_labels[0]} - {MPCs[i]}")

    # Pureza CO2
    axes[2, 0].plot(cycles, y[i][:, 1], label=f"{y_labels[1]} - {MPCs[i]}")

    # Recuperação CO2
    axes[2, 1].plot(cycles, y[i][:, 2], label=f"{y_labels[2]} - {MPCs[i]}")


# ---- Labels ----
axes[0, 0].set_ylabel("Duração / s")
axes[0, 1].set_ylabel("Duração / s")
axes[1, 0].set_ylabel("Duração / s")

axes[1, 1].set_ylabel("KPI / %")
axes[2, 0].set_ylabel("KPI / %")
axes[2, 1].set_ylabel("KPI / %")

for ax in axes.flat:
    ax.grid(True)
    ax.legend()
    ax.set_xlabel("Ciclo")

plt.tight_layout()
plt.savefig("../figures/compare_MPCs.png")
plt.close()


In [3]:
fig, axes = plt.subplots(2, 3, figsize=(18 * 1.1, 9 * 1.1), dpi=300, sharex=True)

# SP
axes[1, 0].plot(cycles, y_sp[0][:, 0], **setpoint_args)
axes[1, 1].plot(cycles, y_sp[0][:, 1], **setpoint_args)
axes[1, 2].plot(cycles, y_sp[0][:, 2], **setpoint_args)

for i, model_name in enumerate(MPCs):
    # u
    axes[0, 0].plot(
        cycles, u[i][:, 0], linewidth=2, label=f"{u_labels[0]} - {model_name}"
    )
    axes[0, 0].set_ylabel("Duração / s")

    axes[0, 1].plot(
        cycles, u[i][:, 1], linewidth=2, label=f"{u_labels[1]} - {model_name}"
    )

    axes[0, 2].plot(
        cycles, u[i][:, 2], linewidth=2, label=f"{u_labels[2]} - {model_name}"
    )
    axes[0, 2].plot(
        cycles, u[i][:, 3], linewidth=2, label=f"{u_labels[3]} - {model_name}"
    )

    # y
    axes[1, 0].plot(
        cycles, y[i][:, 0], linewidth=2, label=f"{y_labels[0]} - {model_name}"
    )
    axes[1, 0].set_ylabel("KPI / %")

    axes[1, 1].plot(
        cycles, y[i][:, 1], linewidth=2, label=f"{y_labels[1]} - {model_name}"
    )

    axes[1, 2].plot(
        cycles, y[i][:, 2], linewidth=2, label=f"{y_labels[2]} - {model_name}"
    )


for ax in axes.flat:
    ax.grid(True)
    ax.set_xlabel("Ciclo")
    ax.legend()

plt.tight_layout(rect=(0.0, 0.0, 1.0, 0.95))
plt.savefig("../figures/compare_MPCs_slide.png")
plt.close()


In [4]:
_, axes = plt.subplots(2, 1, sharex=True, figsize=(12, 8))

for i in range(len(MPCs)):
    axes[i].plot(cycles, Jk[i], label=f"{MPCs[i]}")

axes[-1].set_xlabel("Ciclo")
for ax in axes:
    ax.legend()
    ax.grid(True)
    ax.set_ylabel("Função Custo")

plt.savefig("../figures/cost-functions.png")
plt.close()


# Métricas


In [5]:
e = y - y_sp

# Integral Absolute Error
IAE = np.sum(np.abs(e), axis=1)

# Integral Square Error
ISE = np.sum(e**2, axis=1)


def print_metric_table(metric, metric_name, controllers):
    header = ["Controlador", *y_labels, "Total"]

    n_cols = len(header)
    row_fmt = "{:<14}" + " {:>22}" * (n_cols - 2) + " {:>15}"

    print(f"\n{metric_name}")
    print(row_fmt.format(*header))
    print("-" * (14 + 22 * (n_cols - 2) + 15 + (n_cols - 1)))

    for name, values in zip(controllers, metric):
        total = sum(values)
        print(
            row_fmt.format(
                name,
                *[f"{v:.4f}" for v in values],
                f"{total:.4f}",
            )
        )


print_metric_table(IAE, "IAE", MPCs)
print_metric_table(ISE, "ISE", MPCs)



IAE
Controlador           Pureza do H$_2$       Pureza do CO$_2$  Recuperação de CO$_2$           Total
---------------------------------------------------------------------------------------------------
incremental                   26.3969                45.2334                22.6979         94.3282
IHMPC                         12.0353                32.1269                28.6527         72.8149

ISE
Controlador           Pureza do H$_2$       Pureza do CO$_2$  Recuperação de CO$_2$           Total
---------------------------------------------------------------------------------------------------
incremental                   10.3547               106.6367                36.6850        153.6764
IHMPC                          6.2921                77.6338                56.2245        140.1505
